# Resume / JD Parsing v0

Task 12, Week 4, Phase 2. The sprint title is "E-Sign Integration & Tamper-Evidence", but
the AI/ML deliverable for this task is unrelated to e-sign - it's the start of structured
resume and job-description parsing, so Task 7's matching engine can be fed real extracted
skills instead of someone typing them in by hand.

Order of work:
1. Look at the data and the two traps deliberately built into it
2. Baseline parser (naive substring search) and where it breaks
3. Hardened parser (word-boundary regex + aliases + context filtering)
4. Metrics: baseline vs hardened, on held-out data
5. The evidence - real false positives that got fixed
6. End-to-end demo: resume + JD -> structured data -> match score


In [ ]:
import sys
sys.path.append('../parser')

import pandas as pd
import json

from baseline_parser import baseline_extract_skills
from resume_parser import parse_resume, extract_skills
from jd_parser import parse_jd
from evaluate import evaluate_resumes, evaluate_jds, false_positive_audit_resumes

pd.set_option('display.max_colwidth', 80)


## 1. The data, and the two traps

60 synthetic resumes and 40 job descriptions, free text (not clean JSON), built against a
shared skills ontology (`parser/skills_ontology.py` - the upstream dependency for this task).
Two things are deliberately baked in:

- **substring collisions** - "JavaScript" contains "Java" as a substring, so a resume that
  only knows JavaScript will trip a naive "is 'Java' in this text" check
- **aspirational mentions** - "currently learning Kubernetes" names a skill the candidate
  does NOT have yet, but a naive parser that just checks "does the word appear" will extract it anyway


In [ ]:
resumes_gt = pd.read_csv('../data/resumes_ground_truth.csv')
jds_gt = pd.read_csv('../data/job_descriptions_ground_truth.csv')
print(resumes_gt.shape, jds_gt.shape)
resumes_gt.head()


In [ ]:
print(open('../data/resumes/R0007.txt').read())


This candidate's true skills never include "Java" - only JavaScript. Worth keeping in mind for the next cell.

## 2. Baseline: naive substring search


In [ ]:
text = open('../data/resumes/R0007.txt').read()
print('baseline extracted:', baseline_extract_skills(text))


`Java` shouldn't be there - it's an artifact of "Java" being a substring of "JavaScript". Also note `R` showing up, purely because the letter "r" appears inside ordinary words like "Recent" and "years" - a single-letter skill name is a minefield for substring matching.

In [ ]:
text2 = open('../data/resumes/R0003.txt').read()
print(text2)
print('baseline extracted:', baseline_extract_skills(text2))


`JIRA` gets extracted, but the actual sentence says the candidate is just *interested* in it - not a real skill they have.

## 3. Hardened parser

Two fixes, both in `parser/resume_parser.py`:

1. Word-boundary regex matching (with aliases resolved to one canonical name) instead of
   plain substring search - this alone fixes the Java/JavaScript collision.
2. A small list of aspirational/negative cue phrases checked in the sentence a skill mention
   appears in - "currently learning", "interested in", "no experience with", etc. A skill
   mentioned only inside one of these sentences gets excluded, with the reason kept for
   the explanation layer.


In [ ]:
print('hardened R0007:', extract_skills(text))
print('hardened R0003:', extract_skills(text2))

skills, excluded = extract_skills(text2, return_debug=True)
print('excluded mentions:', excluded)


Both fixed - no `Java` where there shouldn't be one, no `JIRA` from an aspirational sentence, and the exclusion comes with a plain-English reason rather than just silently dropping it.

## 4. Metrics: baseline vs hardened, held-out

Framed as multi-label classification: for every document and every skill in the ontology,
true label = is it actually in ground truth, predicted label = did the parser extract it.
Same precision/recall/false-positive-rate framing used throughout this whole task series.


In [ ]:
rows = []
for split_name in ['dev', 'test']:
    r_split = resumes_gt[resumes_gt['split'] == split_name]
    j_split = jds_gt[jds_gt['split'] == split_name]
    rows.append({'document_type': 'resume', 'model': 'baseline', 'split': split_name,
                 **evaluate_resumes(r_split, baseline_extract_skills)})
    rows.append({'document_type': 'resume', 'model': 'hardened', 'split': split_name,
                 **evaluate_resumes(r_split, parse_resume)})
    rows.append({'document_type': 'jd', 'model': 'baseline', 'split': split_name,
                 **evaluate_jds(j_split, use_baseline=True)})
    rows.append({'document_type': 'jd', 'model': 'hardened', 'split': split_name,
                 **evaluate_jds(j_split, use_baseline=False)})

metrics_df = pd.DataFrame(rows)
metrics_df[['document_type', 'model', 'split', 'precision', 'recall', 'fpr']]


Recall is 1.0 for both baseline and hardened - the baseline doesn't miss real skills, it
invents extra ones. Precision and false-positive rate are where the improvement actually
shows up, and it holds on the held-out test split, not just dev.


## 5. The evidence - real false positives, fixed


In [ ]:
test_resumes = resumes_gt[resumes_gt['split'] == 'test']
audit_df = false_positive_audit_resumes(test_resumes)
print(f"{len(audit_df)} baseline false positives in the test split")
print(f"{int(audit_df['fixed'].sum())} of those are no longer extracted by the hardened parser")
audit_df.head(10)


## 6. End-to-end demo: resume + JD -> structured data -> match score

This is the actual point of the task, per Step 10 of the study guide - parsed output flowing
into a match, with no manual skill entry anywhere in between.


In [ ]:
resume_text = open('../data/resumes/R0001.txt').read()
jd_text = open('../data/job_descriptions/J0001.txt').read()

print('--- RESUME ---')
print(resume_text)
print('--- JOB DESCRIPTION ---')
print(jd_text)


In [ ]:
resume_parsed = parse_resume(resume_text, name='Kavya Kulkarni')
jd_parsed = parse_jd(jd_text)

print('Parsed resume:', json.dumps(resume_parsed, indent=2))
print()
print('Parsed JD:', json.dumps(jd_parsed, indent=2))


In [ ]:
resume_skills = set(resume_parsed['skills'])
required = set(jd_parsed['required_skills'])
nice = set(jd_parsed['nice_to_have_skills'])

matched = resume_skills & required
missing = required - resume_skills
score = len(matched) / len(required) if required else 1.0

print(f"Required-skill match score: {score:.0%}")
print(f"Matched: {sorted(matched)}")
print(f"Missing: {sorted(missing)}")


Two free-text documents in, a structured match score out - the whole point of "Parsing v0
produces structured skills" from the Definition of Done. The structured outputs for the
full dataset are also persisted to `outputs/parsed_resumes.json` and `outputs/parsed_jobs.json`
via `parser/save_outputs.py`, ready for whichever team picks this up next (per the
"Structured profiles/jobs" hand-off in Section 10).
